# Seed Neo4j Inventra: projeção de cobertura de fornecedores (J&F)

Reconstrução completa da projeção de leitura descrita em `neo4j/docs/COBERTURA_FORNECEDORES.md` (seção 06): lê o PostgreSQL (fonte da verdade) e projeta `Requisition`, `Product` e `Supplier` com as arestas `REQUESTS` e `SUPPLIES` no Neo4j.

Usa os mesmos comandos de `neo4j/projection/*.cypher` que o worker da API executa a cada evento, então rodar este notebook equivale a reprocessar todos os eventos. Apaga **só** a projeção no Neo4j; o PostgreSQL nunca é alterado.

Rode o seed do PostgreSQL (`postgre/seeds/seed.ipynb`) antes. Variáveis no `.env`: `DB_*` (PostgreSQL) e `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

# IMPORTS E CONFIGURAÇÃO INICIAL

In [ ]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
from decimal import Decimal
from pathlib import Path
import psycopg2
import os

load_dotenv()

# Os .cypher ficam em neo4j/: funciona rodando de neo4j/seeds (Jupyter) ou da raiz do repositório
NEO4J_DIR = Path.cwd().parent if Path.cwd().name == "seeds" else Path.cwd() / "neo4j"
if not (NEO4J_DIR / "projection").is_dir():
    raise RuntimeError(f"Pasta neo4j/ não encontrada a partir de {Path.cwd()}")

print("Bibliotecas carregadas")

# CONEXÃO COM O POSTGRESQL (fonte da verdade)

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require"),
)
cursor = conn.cursor()

print("Conectado ao PostgreSQL")

# CONEXÃO COM O NEO4J (projeção)

In [ ]:
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "")),
)
driver.verify_connectivity()
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")


def cypher(path):
    """Comandos de um arquivo .cypher (separados por ; no fim da linha)."""
    text = (NEO4J_DIR / path).read_text(encoding="utf-8").replace("\r\n", "\n")
    return [s.strip() for s in text.split(";\n") if s.strip().rstrip(";").strip() and not all(
        l.strip().startswith("//") or not l.strip() for l in s.splitlines())]


def run(path, **params):
    for statement in cypher(path):
        driver.execute_query(statement.rstrip(";"), params, database_=DATABASE)


def num(v):
    """DECIMAL do PostgreSQL vira float no grafo (o driver não serializa Decimal)."""
    return float(v) if isinstance(v, Decimal) else v


print(f"Conectado ao Neo4j: banco '{DATABASE}'")

# SCHEMA (V001)

In [ ]:
for migration in sorted((NEO4J_DIR / "migrations").glob("V*.cypher")):
    run(migration.relative_to(NEO4J_DIR))
    print(f"✅ {migration.name} aplicada")

# RESET DA PROJEÇÃO

Apaga os nós e arestas da projeção (não o schema e nunca o PostgreSQL).

In [ ]:
driver.execute_query(
    "MATCH (n) WHERE n:Requisition OR n:Product OR n:Supplier DETACH DELETE n",
    database_=DATABASE,
)
print("Projeção limpa")

# FORNECEDORES

`tb_supplier` → `:Supplier`. CNPJ, e-mail e WhatsApp ficam de fora: a decisão não precisa deles.

In [ ]:
cursor.execute("SELECT id_supplier, legal_name, rating, active FROM tb_supplier ORDER BY id_supplier")
fornecedores = cursor.fetchall()

for id_supplier, legal_name, rating, active in fornecedores:
    run("projection/supplier_updated.cypher",
        supplierId=id_supplier, legalName=legal_name, rating=rating, active=active)

print(f"✅ {len(fornecedores)} fornecedores projetados")

# PRODUTOS

In [ ]:
cursor.execute("SELECT id_product, name, active FROM tb_product ORDER BY id_product")
produtos = cursor.fetchall()

for id_product, name, active in produtos:
    run("projection/product_updated.cypher", productId=id_product, name=name, active=active)

print(f"✅ {len(produtos)} produtos projetados")

# CATÁLOGO DE FORNECIMENTO

`tb_product_supplier` → `(:Supplier)-[:SUPPLIES]->(:Product)` com código, preço de referência e prazo.

In [ ]:
cursor.execute("""
    SELECT ps.id_supplier, s.legal_name, s.rating, s.active,
           ps.id_product, ps.supplier_code, ps.reference_price, ps.lead_time_days
    FROM tb_product_supplier ps
    JOIN tb_supplier s ON s.id_supplier = ps.id_supplier
    ORDER BY ps.id_supplier, ps.id_product
""")
vinculos = cursor.fetchall()

for id_supplier, legal_name, rating, active, id_product, code, price, lead_time in vinculos:
    run("projection/product_supplier_upserted.cypher",
        supplierId=id_supplier, legalName=legal_name, rating=rating, active=active,
        productId=id_product, supplierCode=code, referencePrice=num(price), leadTimeDays=lead_time)

print(f"✅ {len(vinculos)} vínculos SUPPLIES projetados")

# REQUISIÇÕES E ITENS

`tb_requisition` + `tb_requisition_item` → `(:Requisition)-[:REQUESTS]->(:Product)`, um evento `REQUISITION_UPSERTED` por requisição com a lista completa de itens.

In [ ]:
cursor.execute("""
    SELECT id_requisition, id_kitchen, requisition_type, status
    FROM tb_requisition
    ORDER BY id_requisition
""")
requisicoes = cursor.fetchall()

cursor.execute("""
    SELECT i.id_requisition, i.id_requisition_item, i.id_product, p.name, p.active,
           i.quantity, i.estimated_price, i.note
    FROM tb_requisition_item i
    JOIN tb_product p ON p.id_product = i.id_product
    ORDER BY i.id_requisition, i.id_requisition_item
""")
itens_por_requisicao = {}
for id_req, id_item, id_product, name, active, qty, price, note in cursor.fetchall():
    itens_por_requisicao.setdefault(id_req, []).append({
        "itemId": id_item, "productId": id_product, "productName": name, "productActive": active,
        "quantity": num(qty), "estimatedPrice": num(price), "note": note,
    })

for id_req, id_kitchen, req_type, status in requisicoes:
    run("projection/requisition_upserted.cypher",
        requisition={"id": id_req, "kitchenId": id_kitchen, "type": req_type, "status": status},
        items=itens_por_requisicao.get(id_req, []))

total_itens = sum(len(v) for v in itens_por_requisicao.values())
print(f"✅ {len(requisicoes)} requisições e {total_itens} arestas REQUESTS projetadas")

# RECONCILIAÇÃO

Mesma checagem do job noturno (seção 06): contagens do PostgreSQL × Neo4j.

In [ ]:
def pg_count(sql):
    cursor.execute(sql)
    return cursor.fetchone()[0]


def graph_count(query):
    records, _, _ = driver.execute_query(query, database_=DATABASE)
    return records[0][0]


comparacao = [
    ("Requisition", pg_count("SELECT count(*) FROM tb_requisition"), graph_count("MATCH (n:Requisition) RETURN count(n)")),
    ("Product", pg_count("SELECT count(*) FROM tb_product"), graph_count("MATCH (n:Product) RETURN count(n)")),
    ("Supplier", pg_count("SELECT count(*) FROM tb_supplier"), graph_count("MATCH (n:Supplier) RETURN count(n)")),
    ("REQUESTS", pg_count("SELECT count(*) FROM tb_requisition_item"), graph_count("MATCH ()-[r:REQUESTS]->() RETURN count(r)")),
    ("SUPPLIES", pg_count("SELECT count(*) FROM tb_product_supplier"), graph_count("MATCH ()-[r:SUPPLIES]->() RETURN count(r)")),
]

divergentes = 0
for nome, pg, grafo in comparacao:
    ok = pg == grafo
    divergentes += not ok
    print(f"{'✅' if ok else '❌'} {nome:<12} PostgreSQL {pg:>6}  ·  Neo4j {grafo:>6}")

if divergentes:
    raise RuntimeError(f"{divergentes} contagem(ns) divergente(s) entre PostgreSQL e Neo4j")

# EXEMPLO: COBERTURA DE UMA REQUISIÇÃO EM ANÁLISE

Roda `queries/supplier_coverage.cypher` na requisição `UNDER_REVIEW` com mais itens.

In [ ]:
cursor.execute("""
    SELECT r.id_requisition, r.id_kitchen, count(*) AS itens
    FROM tb_requisition r
    JOIN tb_requisition_item i ON i.id_requisition = r.id_requisition
    WHERE r.status = 'UNDER_REVIEW'
    GROUP BY r.id_requisition, r.id_kitchen
    ORDER BY itens DESC, r.id_requisition
    LIMIT 1
""")
exemplo = cursor.fetchone()

if not exemplo:
    print("Nenhuma requisição UNDER_REVIEW com itens para exemplificar")
else:
    id_req, id_kitchen, n_itens = exemplo
    query = cypher("queries/supplier_coverage.cypher")[0].rstrip(";")
    ranking, _, _ = driver.execute_query(query, requisitionId=id_req, kitchenId=id_kitchen, database_=DATABASE)
    print(f"Requisição #{id_req} (cozinha {id_kitchen}, {n_itens} itens): {len(ranking)} fornecedores candidatos\n")
    for pos, row in enumerate(ranking[:10], 1):
        lacunas = ", ".join(u["name"] for u in row["uncovered"]) or "nenhuma"
        print(f"{pos:>2}. {row['supplierName']:<40} {row['coveragePercent']:>5}%  "
              f"R$ {row['estimatedCoveredCost']:>10.2f}  {row['maxLeadTimeDays']} dias  · falta: {lacunas}")

# FINALIZAR

In [ ]:
cursor.close()
conn.close()
driver.close()

print("Projeção concluída")
print(f"  {len(fornecedores)} fornecedores · {len(produtos)} produtos · {len(vinculos)} vínculos SUPPLIES")
print(f"  {len(requisicoes)} requisições · {total_itens} arestas REQUESTS")